In [ ]:
import anndata as ad
import dask.dataframe as dd
import geopandas as gpd
import pandas as pd
import spatialdata
from shapely import affinity
from spatialdata._io.format import SpatialDataContainerFormatV01
from spatialdata.models import PointsModel, ShapesModel, TableModel
from spatialdata.transformations import Identity, Sequence, get_transformation, set_transformation

# ROI: geojson exported from Xenium Explorer
ROIS = {
    "C-ion_Late": "geojson/Cion_Late_xenium_explorler_annotation.geojson",
    "Xray_Late": "geojson/Xray_Late_xenium_explorler_annotation.geojson",
    "SF1": "geojson/SF1_xenium_explorler_annotation.geojson",
    "SF2": "geojson/SF2_xenium_explorler_annotation.geojson",
    "Xray_Under": "geojson/Xray_Under_xenium_explorler_annotation.geojson",
}
OUT = "integrated/integrated_human.zarr"


def matrix(t):
    return t.to_affine_matrix(input_axes=("x", "y"), output_axes=("x", "y"))


def affine(geom, m):
    return affinity.affine_transform(geom, [m[0, 0], m[0, 1], m[1, 0], m[1, 1], m[0, 2], m[1, 2]])

In [ ]:
tables, cells, nuclei, transcripts, images = {}, {}, {}, {}, {}
outside = []
for roi, geojson in ROIS.items():
    sdata = spatialdata.read_zarr(f"roi_zarrs/human/{roi}.zarr")
    sdata["table"].obs_names = sdata["table"].obs["cell_id"].values
    tables[roi] = sdata["table"]
    cells[roi] = sdata["cell_boundaries"].add_suffix(f"_{roi}", axis=0)
    nuclei[roi] = sdata["nucleus_boundaries"].add_suffix(f"_{roi}", axis=0)
    transcripts[roi] = sdata["transcripts"].assign(roi_id=roi)

    to_um = get_transformation(sdata["cell_boundaries"]).inverse()
    image = spatialdata.get_pyramid_levels(sdata["morphology_focus"], n=0)
    set_transformation(image, Sequence([get_transformation(image), to_um]), "global")
    images[f"morphology_focus_{roi}"] = image

    polygon = affine(gpd.read_file(geojson).geometry[0], matrix(to_um))
    outside += list(cells[roi].index[~cells[roi].centroid.within(polygon).values])

In [ ]:
table = ad.concat(tables, label="roi_id", index_unique="_", merge="same")
# cells in more than one ROI, and cells whose centroid is outside the ROI
drop = table.obs["cell_id"].duplicated(keep=False).values | table.obs_names.isin(outside)
print(pd.crosstab(table.obs["roi_id"], drop, colnames=["dropped"]))

dropped = table.obs_names[drop]
table = table[~drop].copy()
table.obs["region"] = "cell_boundaries"
table.obs["cell_key"] = table.obs_names
table = TableModel.parse(table, region="cell_boundaries", region_key="region", instance_key="cell_key")

cell_boundaries = ShapesModel.parse(pd.concat(cells.values()).loc[table.obs_names])
nucleus_boundaries = pd.concat(nuclei.values())
nucleus_boundaries = ShapesModel.parse(nucleus_boundaries[nucleus_boundaries.index.isin(table.obs_names)])

In [ ]:
transcripts = dd.concat(list(transcripts.values()))
key = transcripts["cell_id"].astype("string") + "_" + transcripts["roi_id"].astype("string")
transcripts = PointsModel.parse(
    transcripts[~key.isin(list(dropped))],
    coordinates={"x": "x", "y": "y", "z": "z"},
    feature_key="feature_name",
    instance_key="cell_id",
)

for element in [cell_boundaries, nucleus_boundaries, transcripts]:
    set_transformation(element, {"global": Identity()}, set_all=True)

In [ ]:
sdata = spatialdata.SpatialData(
    images=images,
    shapes={"cell_boundaries": cell_boundaries, "nucleus_boundaries": nucleus_boundaries},
    points={"transcripts": transcripts},
    tables={"table": table},
)
sdata.write(OUT, sdata_formats=[SpatialDataContainerFormatV01()])
spatialdata.read_zarr(OUT)